In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("/workspaces/SebastianBot")

In [2]:
from cloud.functions.infrastructure.google.helper import load_google_credentials

credentials = load_google_credentials()

In [6]:
from datetime import datetime, timezone, timedelta

from sebastian.shared.gmail.query_builder import GmailQueryBuilder


time_threshold = datetime.now(timezone.utc) - timedelta(hours=720)

query = (
    GmailQueryBuilder()
    .from_email("pickup-point@amazon.de")
    .subject("Paket zur Abholung bereit", exact=False)
    .after_date(time_threshold)
    .build()
)

In [7]:
from sebastian.clients.google.gmail.client import GmailClient


client = GmailClient(credentials=credentials)

In [8]:
mails = client.fetch_mails(query)
mails

[FullMailResponse(id='19d1fd9cfb4b7beb', threadId='19d1fd91ebe2262c', labelIds=['IMPORTANT', 'CATEGORY_PERSONAL', 'INBOX'], snippet='Paket zur Abholung bereit: DHL Packstation 158 Abholung am 31. März͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏ \u200c ͏', sizeEstimate=116352, historyId='8726649', internalDate='1774355860000', content='<!doctype html><html lang="de" dir="auto" xmlns="http://www.w3.org/1999/xhtml" xmlns:v="urn:schemas-microsoft-com:vml" xmlns:o="urn:schemas-microsoft-com:office:office"><head><title></title><!--[if !mso]><!--><meta http-equiv="X-UA-Compatible" content="IE=edge"><!--<![endif]--><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1"><style

In [11]:
from bs4 import BeautifulSoup


mail = mails[0].content
soup = BeautifulSoup(mail, "html.parser")
text = soup.get_text()

In [12]:
text

'Paket zur Abholung bereit: DHL Packstation 158 Abholung am 31. März͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200c \xa0 \u2007 \xad͏ \u200

In [ ]:
from datetime import date

from pydantic import BaseModel, Field


class PickupData(BaseModel):
    tracking_number: str
    pickup_location: str = Field(
        description="When location is Packstation, only include 'Packstation <Number>' as location."
    )
    due_date: date
    item: str = Field(
        description="Description of the item to be picked up, might be truncated"
    )

In [21]:
from cloud.dependencies.clients import resolve_gemini_client


gemini_client = resolve_gemini_client()

In [29]:
prompt = f"""Given the following email text, extract all relevant information:
--- email text start ---
{text}
--- email text end ---
Current year is {date.today().year}
"""

# Use Gemini to parse the email
gemini_client.get_response(prompt, response_schema=PickupData)

PickupData(tracking_number='JJD000390016898240196', pickup_location='DHL Packstation 158', due_date=datetime.date(2026, 3, 31), item='Medela Contact Brusthütchen aus S...')